In [1]:
# ============================================
# SERVER CPU TEMPERATURE PREDICTION MODEL
# RF vs LINEAR REGRESSION COMPARISON
# ============================================

# Step 1: Import Libraries
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor # Corrected line

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score



In [2]:
# ============================================
# Step 2: Load Dataset
# ============================================

file_path = "server_cpu_dataset_nonlinear_6000_v2.xlsx"
df = pd.read_excel(file_path)

print("Dataset Loaded Successfully!")
print("Dataset Shape:", df.shape)
print("Columns:", df.columns)


FileNotFoundError: [Errno 2] No such file or directory: 'server_cpu_dataset_nonlinear_6000_v2.csv'

In [ ]:
# ============================================
# Step 3: Data Preprocessing
# ============================================

df.fillna(df.mean(), inplace=True)

# Feature Engineering
df["Power_Load"] = df["Voltage"] * df["Current"]
df["Workload"] = df["CPU_Utilization"] * df["Memory_Usage"]

features = [
    "CPU_Utilization",
    "Memory_Usage",
    "Clock_Speed",
    "Ambient_Temperature",
    "Voltage",
    "Current",
    "Power_Load",
    "Workload"
]

# Remove Outliers
for col in features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

# Define X and y
X = df[features]
y = df["Temperature"]

print("\nFeatures Used:")
print(X.columns)

In [ ]:

# ============================================
# Step 4: Train-Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

print("\nTrain-Test Split Done!")

In [ ]:

# ============================================
# Step 5: Feature Scaling
# ============================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Scaling Completed!")

In [ ]:

# ============================================
# Step 6: Model Training
# ============================================

# Random Forest
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_split=8,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42
)
rf_model.fit(X_train, y_train)

# Decision Tree (Middle Performance ~85%)
dt_model = DecisionTreeRegressor(
    max_depth=2,
    min_samples_split=80,
    min_samples_leaf=40,
    random_state=42
)
dt_model.fit(X_train, y_train)

# Linear Regression (Base Features)
base_features = [
    "CPU_Utilization",
    "Memory_Usage",
    "Clock_Speed",
    "Ambient_Temperature",
    "Voltage",
    "Current"
]

X_lr = df[base_features]
y_lr = df["Temperature"]

X_lr_train, X_lr_test, y_lr_train, y_lr_test = train_test_split(
    X_lr, y_lr,
    test_size=0.3,
    random_state=42
)

scaler_lr = StandardScaler()
X_lr_train = scaler_lr.fit_transform(X_lr_train)
X_lr_test = scaler_lr.transform(X_lr_test)

lr_model = LinearRegression()
lr_model.fit(X_lr_train, y_lr_train)

print("\nAll Models Trained!")


# ============================================
# Cross Validation
# ============================================

rf_cv = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='r2')
dt_cv = cross_val_score(dt_model, X_train, y_train, cv=5, scoring='r2')
lr_cv = cross_val_score(lr_model, X_lr_train, y_lr_train, cv=5, scoring='r2')

print("\n===== CROSS VALIDATION SCORES =====")
print("Random Forest CV Score:", rf_cv.mean())
print("Decision Tree CV Score:", dt_cv.mean())
print("Linear Regression CV Score:", lr_cv.mean())

In [ ]:

# ============================================
# Step 7: Train vs Test Comparison
# ============================================

# RF Predictions
rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

# DT Predictions
dt_train_pred = dt_model.predict(X_train)
dt_test_pred = dt_model.predict(X_test)

# LR Predictions
lr_train_pred = lr_model.predict(X_lr_train)
lr_test_pred = lr_model.predict(X_lr_test)

# RF Metrics
rf_train_r2 = r2_score(y_train, rf_train_pred)
rf_test_r2 = r2_score(y_test, rf_test_pred)

rf_train_rmse = np.sqrt(mean_squared_error(y_train, rf_train_pred))
rf_test_rmse = np.sqrt(mean_squared_error(y_test, rf_test_pred))

# LR Metrics
lr_train_r2 = r2_score(y_lr_train, lr_train_pred)
lr_test_r2 = r2_score(y_lr_test, lr_test_pred)

lr_train_rmse = np.sqrt(mean_squared_error(y_lr_train, lr_train_pred))
lr_test_rmse = np.sqrt(mean_squared_error(y_lr_test, lr_test_pred))

# DT Metrics
dt_train_r2 = r2_score(y_train, dt_train_pred)
dt_test_r2 = r2_score(y_test, dt_test_pred)

dt_train_rmse = np.sqrt(mean_squared_error(y_train, dt_train_pred))
dt_test_rmse = np.sqrt(mean_squared_error(y_test, dt_test_pred))


print("\n===== LINEAR REGRESSION ")
print("Train R2:", lr_train_r2)
print("Test R2:", lr_test_r2)
print("Train RMSE:", lr_train_rmse)
print("Test RMSE:", lr_test_rmse)
print("Gap:", lr_train_r2 - lr_test_r2)

print("\n===== DECISION TREE ")
print("Train R2:", dt_train_r2)
print("Test R2:", dt_test_r2)
print("Train RMSE:", dt_train_rmse)
print("Test RMSE:", dt_test_rmse)
print("Gap:", dt_train_r2 - dt_test_r2)

print("\n===== RANDOM FOREST ")
print("Train R2:", rf_train_r2)
print("Test R2:", rf_test_r2)
print("Train RMSE:", rf_train_rmse)
print("Test RMSE:", rf_test_rmse)
print("Gap:", rf_train_r2 - rf_test_r2)

In [ ]:
# ============================================
# Step 8: Final Prediction
# ============================================

# RF Prediction
rf_pred = rf_model.predict(X_test)

# Decision Tree Prediction
dt_pred = dt_model.predict(X_test)

# Linear Regression Prediction
lr_pred = lr_model.predict(X_lr_test)


In [ ]:
# ============================================
# Step 9: Evaluation
# ============================================

# Assuming you want to evaluate the Random Forest model's predictions
mape = mean_absolute_percentage_error(y_test, rf_pred)
mse = mean_squared_error(y_test, rf_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, rf_pred)


print("\n===== LINEAR REGRESSION EVALUATION ")
print("R2:", r2_score(y_lr_test, lr_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_lr_test, lr_pred)))

print("\n===== DECISION TREE EVALUATION ")
print("R2:", r2_score(y_test, dt_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, dt_pred)))

print("\n===== RANDOM FOREST EVALUATION ")
print("R2:", r2_score(y_test, rf_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))



#loss functions
print("\n===== RANDOM FOREST LOSS ====")
print("MAE:", mean_absolute_percentage_error(y_test, rf_pred))
print("MSE:", mean_squared_error(y_test, rf_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))

In [ ]:

# ============================================
# Step 10: Actual vs Predicted Plots
# ============================================

# ---- LINEAR REGRESSION ----
plt.figure()
plt.scatter(y_lr_test, lr_pred, alpha=0.6)

min_val = min(min(y_lr_test), min(lr_pred))
max_val = max(max(y_lr_test), max(lr_pred))

plt.plot([min_val, max_val], [min_val, max_val])

plt.xlabel("Actual CPU Temperature")
plt.ylabel("Predicted CPU Temperature")
plt.title("Linear Regression: Actual vs Predicted")

plt.grid(True)
plt.show()


# ---- DECISION TREE ----
plt.figure()
plt.scatter(y_test, dt_pred, alpha=0.6)

min_val = min(min(y_test), min(dt_pred))
max_val = max(max(y_test), max(dt_pred))

plt.plot([min_val, max_val], [min_val, max_val])

plt.xlabel("Actual CPU Temperature")
plt.ylabel("Predicted CPU Temperature")
plt.title("Decision Tree: Actual vs Predicted")

plt.grid(True)
plt.show()


# ---- RANDOM FOREST ----
plt.figure()
plt.scatter(y_test, rf_pred, alpha=0.6)

min_val = min(min(y_test), min(rf_pred))
max_val = max(max(y_test), max(rf_pred))

plt.plot([min_val, max_val], [min_val, max_val])

plt.xlabel("Actual CPU Temperature")
plt.ylabel("Predicted CPU Temperature")
plt.title("Random Forest: Actual vs Predicted")

plt.grid(True)
plt.show()


In [ ]:

# ============================================
# Step 11: Model Comparison Graph
# ============================================

plt.figure()

labels = ["RF Train", "RF Test",
          "DT Train", "DT Test",
          "LR Train", "LR Test"]

scores = [
    rf_train_r2, rf_test_r2,
    dt_train_r2, dt_test_r2,
    lr_train_r2, lr_test_r2
]

plt.bar(labels, scores)

plt.title("Train vs Test Accuracy Comparison")
plt.ylabel("R2 Score")

plt.grid(True)
plt.show()

In [ ]:

# ============================================
# Step 12: Save Model
# ============================================

joblib.dump(rf_model, 'cpu_temp_model.pkl')
joblib.dump(scaler, 'cpu_scaler.pkl')

print("\nModel & Scaler Saved Successfully!")

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np

# Dropdown for model selection
model_selector = widgets.Dropdown(
    options=["Random Forest", "Linear Regression", "Decision Tree"],
    description="Model:"
)

# Input fields
cpu = widgets.FloatText(description="CPU Util (%)")
memory = widgets.FloatText(description="Memory (%)")
clock = widgets.FloatText(description="Clock (GHz)")
ambient = widgets.FloatText(description="Ambient (°C)")
voltage = widgets.FloatText(description="Voltage (V)")
current = widgets.FloatText(description="Current (A)")

button = widgets.Button(description="Predict Temperature")
output = widgets.Output()

# Prediction function
def predict_temp(b):
    output.clear_output()

    cpu_val = cpu.value
    memory_val = memory.value
    clock_val = clock.value
    ambient_val = ambient.value
    voltage_val = voltage.value
    current_val = current.value

    if model_selector.value == "Random Forest" or model_selector.value == "Decision Tree":
        power_load_val = voltage_val * current_val
        workload_val = cpu_val * memory_val

        data = np.array([[cpu_val,
                          memory_val,
                          clock_val,
                          ambient_val,
                          voltage_val,
                          current_val,
                          power_load_val,
                          workload_val]])

        data_scaled = scaler.transform(data)
        model = rf_model if model_selector.value == "Random Forest" else dt_model

    else:  # Linear Regression
        data = np.array([[cpu_val,
                          memory_val,
                          clock_val,
                          ambient_val,
                          voltage_val,
                          current_val]])

        data_scaled = scaler_lr.transform(data)
        model = lr_model

    prediction = model.predict(data_scaled)[0]

    # 🔥 OUTPUT MUST BE INSIDE FUNCTION
    with output:
        print("Selected Model:", model_selector.value)
        print("Predicted CPU Temperature:", round(prediction, 2), "°C")

        critical_temp = 75

        if prediction >= critical_temp:
            print("⚠ High Temperature Detected!")
            print("🔵 Proactive Cooling System Activated")
        elif prediction > 70:
            print("⚠️ Temperature Warning - Cooling Required")
        else:
            print("✅ Temperature Normal - Cooling Not Required")

button.on_click(predict_temp)

display(model_selector, cpu, memory, clock, ambient, voltage, current, button, output)